Vamos demonstrar as fases de ETL (Extract, Transformatio and Load)

Utilizaremos duas bases de dados

Extract - nesta fase faremos a extração dos dados.

In [1]:
# Usaremos como base de dados um arquivo csv

import pandas as pd

df_csv = pd.read_csv('dados.csv', sep=";")
print(df_csv.dtypes) # exibe nomes das colunas e seus respectivos tipos de dados

codigo             int64
nome                 str
numero_conta         str
agencia            int64
saldo_conta      float64
limite_conta     float64
numero_cartao        str
limite_cartao    float64
dtype: object


Transformation - nesta fase faremos os tratamentos necessários dos dados

In [2]:
# trataremos a df_csv
# padronizando os nomes
df_csv['nome'] = df_csv['nome'].str.title()

# convertendo o tipo para str e padronizando a quantidade de caracteres
df_csv['agencia'] = df_csv['agencia'].astype(str).str.zfill(4)

# convertendo colunas do tipo float64 para str (padrão da API)
df_csv['saldo_conta'] = df_csv['saldo_conta'].astype(str)
df_csv['limite_conta'] = df_csv['limite_conta'].astype(str)
df_csv['limite_cartao'] = df_csv['limite_cartao'].astype(str)

Utilizaremos a API do Gemini para gerar mensagem de marketing para cada usuário.

In [3]:
# resgatando a api_key
import os
from dotenv import load_dotenv

# carregando dados do .env
load_dotenv()
my_api_key = os.getenv("API_KEY")

print('Chave recuperada.' if my_api_key else 'Chave invalida.')

Chave recuperada.


In [4]:
# iremos criar a coluna 'news' no dataframe
df_csv['news'] = [[] for _ in range(len(df_csv))]
print(df_csv.columns)

Index(['codigo', 'nome', 'numero_conta', 'agencia', 'saldo_conta',
       'limite_conta', 'numero_cartao', 'limite_cartao', 'news'],
      dtype='str')


In [5]:
from google import genai

# inicializando o cliente
client = genai.Client(api_key=my_api_key)

def generate_ai_news(user):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Você é um especialista em marketing bancário. Crie uma mensagem para {user['nome']} sobre a importância dos investimentos com no máximo de 100 caracteres"
    )
    return response.text

# usaremos uma lista para armazenar as notificações

for i, user in df_csv.iterrows():
    texto = generate_ai_news(user)
    news = texto.strip()

    user['news'].append({
        "icon": "icons/mensagem.png",
        "description": news
    })
    print(user['news'])

[{'icon': 'icons/mensagem.png', 'description': 'Ana, não deixe o amanhã ao acaso. Invista agora e garanta um futuro próspero e seguro para você!'}]
[{'icon': 'icons/mensagem.png', 'description': 'Bruno, invista no seu futuro! Seu patrimônio agradece e o tempo potencializa seus ganhos.'}]
[{'icon': 'icons/mensagem.png', 'description': 'Carla, invista em seu futuro! Multiplique seu dinheiro, realize sonhos e construa segurança. Comece já! (95 caracteres)'}]
[{'icon': 'icons/mensagem.png', 'description': 'Diego, invista no seu futuro hoje! Faça seu dinheiro crescer e realize seus sonhos.'}]
[{'icon': 'icons/mensagem.png', 'description': 'Elena, seu futuro merece mais! Invista agora para que seu dinheiro trabalhe por você e seus objetivos.'}]


Load - nesta fase iremos enviar os dados gerados para uma API.

In [6]:
api_url = "http://127.0.0.1:8000"  # neste exemplo utilizaremos uma API local

In [7]:
# preparando os dados para o API
def formatar_para_json(row):
    return {
        "id": int(row['codigo']),
        "name": row['nome'],
        "account": {
            "id": None,
            "number": str(row['numero_conta']),
            "agency": row['agencia'],
            "balance": row['saldo_conta'],
            "limit": row['limite_conta']
        },
        "card": {
            "id": None,
            "number": str(row['numero_cartao']),
            "limit": row['limite_cartao']
        },
        "news": [
            {
                "id": None,
                "icon": str(row['news'][0]['icon']),
                "description": str(row['news'][0]['description'])
            }
        ],
        "features": []
    }

dados = df_csv.apply(formatar_para_json, axis=1).tolist()
print("Dados gerados." if dados else "Falha ao gerar dados.")

Dados gerados.


In [8]:
import requests
# inserindo os dados na API

for registro in dados:
    # Usando o parâmetro 'json=' para garantir o Header correto
    # No seu script do Pandas, altere esta linha:
    response = requests.post(f"{api_url}/users/", json=registro)

    # Use == para comparação
    status = f"{registro['name']} cadastrado com sucesso" if response.status_code == 201 else response.text
    print(status)

Ana Silva cadastrado com sucesso
Bruno Oliveira cadastrado com sucesso
Carla Mendes cadastrado com sucesso
Diego Santos cadastrado com sucesso
Elena Rocha cadastrado com sucesso
